In [ ]:
import csv
import time
import requests
from bs4 import BeautifulSoup

# Normarizar fechas
def normalizar_fecha_escrita(fecha_raw):
    return fecha_raw

# links 
urls = {
    "Vitoria": "https://www.booking.com/reviews/es/hotel/libere-vitoria-centro.es.html",
    "Donosti": "https://www.booking.com/reviews/es/hotel/koisi-hostel.es.html",
    "BilbaoMuseo": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-guggenheim.es.html",
    "BilbaoLaVieja": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-la-vieja.es.html",
    "ValenciaAbastos": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-abastos.es.html",
    "PamplonaYamaguchi": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-pamplona-yamaguchi.es.html",
    "ValenciaJardinBotanico": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-jardin-botanico.es.html",
    "MadridPalacioReal": "https://www.booking.com/reviews/es/hotel/libere-madrid-palacio-real.es.html",
    "MalagaTeatroRomano": "https://www.booking.com/reviews/es/hotel/apartamentosliberemalagateatroromano.es.html",
    "GranadaCatedral": "https://www.booking.com/reviews/es/hotel/apartamentos-libere-granada-catedral.es.html",
    "MalagaLaMerced": "https://www.booking.com/reviews/es/hotel/libere-malaga-la-merced.es.html",
    "CordobaPatio": "https://www.booking.com/reviews/es/hotel/libere-cordoba-patio-santa-marta.es.html"}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "es-ES,es;q=0.9"}

with open("comentarios_booking.csv", mode="w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)

    # Titulos CSV
    writer.writerow([
        "ubicacion",
        "fecha",
        "puntuacion",
        "titulo_comentario",
        "comentario_negativo",
        "comentario_positivo",
        "etiquetas",
        "cantidad_comentarios",
        "nacionalidad"])

    for ubicacion, base_url in urls.items():
        print(f"\nScrapeando {ubicacion}")

        page = 1

        # Revisar todas las paginas
        while True:
            print(f"  Página {page}")

            url = f"{base_url}?page={page}"
            response = requests.get(url, headers=headers)
            # Cambiado de "lxml" a "html.parser"
            soup = BeautifulSoup(response.text, "html.parser")

            contenedor = soup.select("li.review_item.clearfix")

            if not contenedor:
                print("No hay mas")
                break

            for reseña in contenedor:

                # Fecha
                fecha_el = reseña.select_one("p.review_item_date")
                fecha_raw = fecha_el.get_text(strip=True) if fecha_el else ""
                fecha = normalizar_fecha_escrita(fecha_raw)

                # Puntuacion
                score = reseña.select_one("span.review-score-badge")
                puntuacion = score.get_text(strip=True) if score else ""

                # Titulo
                titulo_el = reseña.select_one("span[itemprop='name']")
                titulo = titulo_el.get_text(strip=True) if titulo_el else ""

                # Comentario negativo
                neg = reseña.select_one("p.review_neg span[itemprop='reviewBody']")
                comentario_negativo = neg.get_text(strip=True) if neg else ""

                # Comentario positivo
                pos = reseña.select_one("p.review_pos span[itemprop='reviewBody']")
                comentario_positivo = pos.get_text(strip=True) if pos else ""

                # Etiquetas
                etiquetas = reseña.select("ul.review_item_info_tags li")
                etiquetas_texto = " | ".join(
                    e.get_text(strip=True).replace("•", "").strip()
                    for e in etiquetas)

                # Cantidad de Comentarios
                com = reseña.select_one("div.review_item_user_review_count")
                comentarios_n = com.get_text(strip=True) if neg else ""

                # Cantidad de Comentarios
                nac = reseña.select_one("div.review_item_reviewer span[itemprop='nationality']")
                nacionalidad = nac.get_text(strip=True) if neg else ""

                # Guardar 
                writer.writerow([
                    ubicacion,
                    fecha,
                    puntuacion,
                    titulo,
                    comentario_negativo,
                    comentario_positivo,
                    etiquetas_texto,
                    comentarios_n,
                    nacionalidad
                ])

            page += 1
            time.sleep(1)

print("\n CSV creado correctamente con todas las ubicaciones")


Scrapeando Vitoria
  Página 1
  Página 2
  Página 3
  Página 4
  Página 5
  Página 6
  Página 7
  Página 8
  Página 9
  Página 10
  Página 11
  Página 12
  Página 13
  Página 14
  Página 15
  Página 16
  Página 17
  Página 18
  Página 19
  Página 20
  Página 21
  Página 22
  Página 23
  Página 24
  Página 25
  Página 26
  Página 27
  Página 28
  Página 29
  Página 30
  Página 31
  Página 32
  Página 33
  Página 34
  Página 35
  Página 36
  Página 37
  Página 38
  Página 39
  Página 40
  Página 41
  Página 42
  Página 43
  Página 44
No hay mas

Scrapeando Donosti
  Página 1


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords 
from nltk.stem.porter import PorterStemmer
import re
import math
import numpy as np
import nltk

def clean_text_round(text):
    
    text = re.sub('\r', '', text)
    text = re.sub('\n', ' ', text)
    text = re.sub('Â', '', text)
    text = re.sub('\w*\d\w*', '', text) 
    text = re.sub('\u200a', '', text)
    return text

with open('file_text.txt', encoding='utf-8') as f:
    lines = f.read()

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(lines)
total_documents = len(sentences)

data_df = pd.DataFrame.from_dict(sentences)
data_df.columns = ['Contenido']
data_df.head()

round = lambda x: clean_text_round(x)

data_clean = pd.DataFrame(data_df.Contenido.apply(round))
data_clean

data_clean['Contenido'][0]

def _create_frequency_matrix(sentences):
    frequency_matrix = {}
    stopWords = set(stopwords.words("english"))
    indice = 0
    ps = PorterStemmer()

    for sent in sentences:
        freq_table = {}
        words = word_tokenize(sent)
        for word in words:
            word = word.lower()
            word = ps.stem(word)
            if word in stopWords:
                continue

            if word in freq_table:
                freq_table[word] += 1
            else:
                freq_table[word] = 1

        #frequency_matrix[str(indice)] = freq_table
        frequency_matrix[sent] = freq_table
        #indice += 1
    return frequency_matrix

import nltk
nltk.download('stopwords')
fm = _create_frequency_matrix(data_clean['Contenido'])
fm

def _create_tf_matrix(freq_matrix):
    tf_matrix = {}

    for desc, f_table in freq_matrix.items(): # desc - nº doc, f_table - diccionario de frecuencias
        tf_table = {}

        count_words_in_desc = len(f_table) # Nº de palabras únicas que tenemos en cuenta por doc
        for word, count in f_table.items():
            tf_table[word] = count / count_words_in_desc # Porcentaje de aparición de cada palabra por documento

        tf_matrix[desc] = tf_table

    return tf_matrix

tf_matrix = _create_tf_matrix(fm)

def _create_documents_per_words(freq_matrix):
    word_per_doc_table = {}

    for dok, f_table in freq_matrix.items():
        for word, count in f_table.items():
            if word in word_per_doc_table:
                word_per_doc_table[word] += 1
            else:
                word_per_doc_table[word] = 1

    return word_per_doc_table

d_per_w = _create_documents_per_words(fm)
d_per_w

def _create_idf_matrix(freq_matrix, count_doc_per_words, total_documents): # diccionario de word:freq por contenido; word:freq aparición documentos; contenidos en total
    idf_matrix = {}

    for desc, f_table in freq_matrix.items():
        idf_table = {}

        for word in f_table.keys():
            idf_table[word] = math.log10(total_documents / float(count_doc_per_words[word])) # Implementación de la fórmula

        idf_matrix[desc] = idf_table

    return idf_matrix

idf_m = _create_idf_matrix(fm, d_per_w, total_documents)
idf_m

def _create_tf_idf_matrix(tf_matrix, idf_matrix):
    tf_idf_matrix = {}

    for (desc1, f_table1), (desc2, f_table2) in zip(tf_matrix.items(), idf_matrix.items()):

        tf_idf_table = {}

        for (word1, value1), (word2, value2) in zip(f_table1.items(),
                                                    f_table2.items()):  # here, keys are the same in both the table
            tf_idf_table[word1] = float(value1 * value2)

        tf_idf_matrix[desc1] = tf_idf_table

    return tf_idf_matrix

tf_idf = _create_tf_idf_matrix(tf_matrix, idf_m)
tf_idf

def _score_sentences(tf_idf_matrix):
    """
    score a sentence by its word's TF
    Basic algorithm: adding the TF frequency of every non-stop word in a sentence divided by total no of words in a sentence.
    :rtype: dict
    """

    sentenceValue = {}

    for sent, f_table in tf_idf_matrix.items():
        total_score_per_sentence = 0

        count_words_in_sentence = len(f_table)
        for word, score in f_table.items():
            total_score_per_sentence += score

        sentenceValue[sent] = total_score_per_sentence / count_words_in_sentence

    return sentenceValue

sentence_scores = _score_sentences(tf_idf)
sentence_scores

def _find_average_score(sentenceValue):
    """
    Find the average score from the sentence value dictionary
    :rtype: int
    """
    sumValues = 0
    for entry in sentenceValue:
        sumValues += sentenceValue[entry]

    # Average value of a sentence from original summary_text
    average = (sumValues / len(sentenceValue))

    return average

threshold = _find_average_score(sentence_scores)
threshold

def _generate_summary(sentences, sentenceValue, threshold):
    sentence_count = 0
    summary = ''

    for sentence in sentences:
        if sentence in sentenceValue and sentenceValue[sentence] >= (threshold):
            summary += " " + sentence
            sentence_count += 1

    return summary

summary = _generate_summary(data_clean['Contenido'], sentence_scores, 1.3 * threshold)
print(summary)

fm              = _create_frequency_matrix(data_clean['Contenido'])
tf_matrix       = _create_tf_matrix(fm)
d_per_w         = _create_documents_per_words(fm)
idf_m           = _create_idf_matrix(fm, d_per_w, total_documents)
tf_idf          = _create_tf_idf_matrix(tf_matrix, idf_m)
sentence_scores = _score_sentences(tf_idf)
threshold       = _find_average_score(sentence_scores)
summary         = _generate_summary(data_clean['Contenido'], sentence_scores, 1.3 * threshold)
print(summary)

<>:17: SyntaxWarning: invalid escape sequence '\w'
<>:17: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Umiak\AppData\Local\Temp\ipykernel_11468\1256336765.py:17: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)
C:\Users\Umiak\AppData\Local\Temp\ipykernel_11468\1256336765.py:17: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


FileNotFoundError: [Errno 2] No such file or directory: 'file_text.txt'